First we will get all the data from the Neo4j instance into a polars dataframe.

In [ ]:
from typing import Any, Dict, List, Optional

import polars as pl
from neo4j import AsyncGraphDatabase

# user defined variables for neo4j
URI = "bolt://localhost:7687"
USER = "neo4j" # dev config
PASSWORD = "neo4j" # dev config

# query to pull everything
QUERY = """MATCH (n)
OPTIONAL MATCH (n)-[r]-()
RETURN n, r"""

async def run_match_query(
    uri: str,
    user: str,
    password: str,
    query: str,
    database: Optional[str] = None,
    params: Optional[Dict[str, Any]] = None,
) -> List[Dict[str, Any]]:
    """Connect asynchronously to Neo4j and run a MATCH (or any Cypher) query.

    Returns a list of dictionaries (one per record). Use `to_polars(records)`
    to convert the results to a Polars DataFrame.
    """
    driver = AsyncGraphDatabase.driver(uri, auth=(user, password))
    try:
        async with driver.session(database=database) as session:
            result = await session.run(query, params or {})
            records: List[Dict[str, Any]] = []
            async for record in result:
                records.append(record.data())
            return records
    finally:
        await driver.close()

all_records = await run_match_query(
    uri=URI,
    user=USER,
    password=PASSWORD,
    query=QUERY,
)
df = pl.DataFrame(all_records)
df.head()



arm64
